# Person 2 — Classification & Matching Pipeline

This notebook executes all of Person 2's responsibilities:

| Step | Phase | Task |
|------|-------|------|
| 1 | Setup | Imports, paths, window labels |
| 2 | Phase 4 | Active/Passive set intersections (Jaccard, enrichment) |
| 3 | Phase 5 | Feature extraction validation for overlapping addresses |
| 4 | Phase 3 | Classification pipeline for all window sizes |
| 5 | Phase 6 | Correlation analysis (6a temporal, 6b volume, 6c frequency, 6d ratio) |
| 6 | Export | Save all results to outputs/ for Person 3 |

**Prerequisite:** Person 1 must have run `P1_data_layer_pipeline.ipynb` so that
`data/interim/` and `data/interim/feature_rows/` are populated.

In [ ]:
# ── Step 1: Setup ────────────────────────────────────────────────────────────
import sys, os, logging
from pathlib import Path

# Add src/ to the Python path so we can import our modules
REPO_ROOT = Path("__file__").resolve().parent.parent
SRC_DIR   = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("P2")

# ── Project paths ─────────────────────────────────────────────────────────────
DATA_DIR        = REPO_ROOT / "data"
INTERIM_DIR     = DATA_DIR / "interim"
FEATURE_ROW_DIR = INTERIM_DIR / "feature_rows"
PROCESSED_DIR   = DATA_DIR / "processed"
OUTPUTS_DIR     = REPO_ROOT / "outputs"
TABLES_DIR      = OUTPUTS_DIR / "tables"
FIGURES_DIR     = OUTPUTS_DIR / "figures"

for d in [TABLES_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Window labels as produced by Person 1's updated config.py
WINDOW_LABELS = ["1blk", "10blk", "1000blk", "10000blk"]
CHAINS        = ["ethereum", "polygon", "bnb"]

print("Repo root:", REPO_ROOT)
print("Windows  :", WINDOW_LABELS)
print("Chains   :", CHAINS)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

# Person 2 modules
from normalize import normalize_feature_table, build_cross_chain_address_summary
from classify  import classify_address_summary, classify_from_normalized_df, behavioral_group_counts
from set_ops   import (
    load_address_set,
    build_full_set_intersection_report,
    build_cross_window_set_summary,
    save_set_report,
)
from correlation import (
    load_feature_table,
    run_all_correlations,
    build_cross_window_correlation_summary,
    save_correlation_results,
)

print("All imports OK")

---
## Step 2 — Phase 4: Active/Passive Set Intersections

Loads `active_eoa_addresses.csv` and `passive_eoa_addresses.csv` for each chain × window,
then computes pairwise, triple, and mixed intersections with Jaccard similarity.

In [ ]:
# ── Step 2: Set intersection reports for each window ─────────────────────────
set_reports = {}

for window in WINDOW_LABELS:
    logger.info("Building set intersection report: window=%s", window)
    report = build_full_set_intersection_report(
        interim_dir=INTERIM_DIR,
        window_label=window,
        chains=CHAINS,
    )
    set_reports[window] = report
    save_set_report(report, output_dir=TABLES_DIR, window_label=window)
    print(f"\n── {window} ──")
    display(report["summary"])

In [ ]:
# Cross-window comparison of Jaccard scores
cross_window_sets = build_cross_window_set_summary(set_reports)
cross_window_sets.to_csv(TABLES_DIR / "set_intersections_all_windows.csv", index=False)
display(cross_window_sets.pivot_table(
    index="comparison", columns="window", values="jaccard", aggfunc="first"
))

In [ ]:
# Print triple-intersection sizes across windows
print("Triple intersection sizes (active)")
for window, report in set_reports.items():
    triple = report.get("active_triple", {})
    if triple:
        print(f"  {window}: {triple['intersection_size']} addresses  (Jaccard={triple['jaccard']:.8f})")

print("\nTriple intersection sizes (passive)")
for window, report in set_reports.items():
    triple = report.get("passive_triple", {})
    if triple:
        print(f"  {window}: {triple['intersection_size']} addresses  (Jaccard={triple['jaccard']:.8f})")

---
## Step 3 — Phase 5: Feature extraction validation for overlapping addresses

Verifies that per-address per-chain features exist for every address present on 2+ networks.

In [ ]:
# ── Step 3: Validate feature coverage for overlapping addresses ───────────────
validation_rows = []

for window in WINDOW_LABELS:
    # Load feature table for this window
    feat_csv = FEATURE_ROW_DIR / f"address_chain_features_{window}.csv"
    if not feat_csv.exists():
        logger.warning("Feature CSV not found: %s — skipping", feat_csv)
        continue

    df_feat = pd.read_csv(feat_csv)
    total_records     = len(df_feat)
    unique_addresses  = df_feat["address"].nunique()
    unique_chains     = df_feat["chain"].nunique()

    # Addresses present on 2+ chains
    chain_counts      = df_feat.groupby("address")["chain"].nunique()
    multi_chain_addrs = (chain_counts >= 2).sum()
    tri_chain_addrs   = (chain_counts >= 3).sum()

    # Missing feature columns
    key_cols = [
        "first_seen_ts", "last_seen_ts", "total_tx_count",
        "recent_7d_tx_count", "recent_30d_tx_count", "recent_90d_tx_count",
        "active_days", "activity_density",
    ]
    missing_cols = [c for c in key_cols if c not in df_feat.columns]

    validation_rows.append({
        "window": window,
        "feature_records": total_records,
        "unique_addresses": unique_addresses,
        "chains_covered": unique_chains,
        "multi_chain_addresses": multi_chain_addrs,
        "tri_chain_addresses": tri_chain_addrs,
        "missing_key_cols": missing_cols if missing_cols else "none",
    })
    logger.info(
        "%s: %d records | %d unique addrs | %d on 2+ chains | missing cols: %s",
        window, total_records, unique_addresses, multi_chain_addrs,
        missing_cols or "none",
    )

df_validation = pd.DataFrame(validation_rows)
df_validation.to_csv(TABLES_DIR / "p2_feature_validation.csv", index=False)
display(df_validation)

---
## Step 4 — Classification for all window sizes

Runs the full feature → normalize → classify pipeline for each window.

In [ ]:
# ── Step 4: Classify for every window ────────────────────────────────────────
classified_tables = {}

for window in WINDOW_LABELS:
    feat_csv = FEATURE_ROW_DIR / f"address_chain_features_{window}.csv"
    if not feat_csv.exists():
        logger.warning("Skipping classification for %s — feature CSV not found", window)
        continue

    logger.info("Classifying window=%s", window)
    df_feat = pd.read_csv(feat_csv)

    # Normalize → summarize → classify
    df_classified = classify_from_normalized_df(df_feat)
    classified_tables[window] = df_classified

    # Save
    out_csv = TABLES_DIR / f"classified_{window}.csv"
    out_pq  = PROCESSED_DIR / window / f"classified_{window}.parquet"
    out_pq.parent.mkdir(parents=True, exist_ok=True)

    df_classified.to_csv(out_csv, index=False)
    df_classified.to_parquet(out_pq, index=False)

    counts = behavioral_group_counts(df_classified)
    print(f"\n── {window} ── ({len(df_classified)} addresses)")
    display(counts)

In [ ]:
# Cross-window behavioral group shares
summary_rows = []
for window, df_cl in classified_tables.items():
    counts = behavioral_group_counts(df_cl)
    for _, row in counts.iterrows():
        summary_rows.append({
            "window": window,
            "behavioral_group": row["behavioral_group"],
            "count": row["count"],
            "share": row["share"],
        })

df_class_summary = pd.DataFrame(summary_rows)
df_class_summary.to_csv(TABLES_DIR / "classification_summary_all_windows.csv", index=False)
display(df_class_summary.pivot_table(
    index="behavioral_group", columns="window", values="share", aggfunc="first"
).round(4))

---
## Step 5 — Phase 6: Correlation Analysis

Runs all four correlation types (6a–6d) for each window size.

In [ ]:
# ── Step 5: Correlation analysis per window ───────────────────────────────────
all_corr_results = {}

for window in WINDOW_LABELS:
    feat_csv = FEATURE_ROW_DIR / f"address_chain_features_{window}.csv"
    if not feat_csv.exists():
        logger.warning("Skipping correlations for %s — feature CSV not found", window)
        continue

    logger.info("Running correlations: window=%s", window)
    df_feat = pd.read_csv(feat_csv)

    # Load active/passive sets for 6d ratio analysis
    active_sets  = {}
    passive_sets = {}
    for chain in CHAINS:
        chain_dir = INTERIM_DIR / window / chain
        active_sets[chain]  = load_address_set(chain_dir / "active_eoa_addresses.csv")
        passive_sets[chain] = load_address_set(chain_dir / "passive_eoa_addresses.csv")

    results = run_all_correlations(
        feature_df=df_feat,
        active_sets=active_sets,
        passive_sets=passive_sets,
        chains=CHAINS,
    )
    all_corr_results[window] = results

    save_correlation_results(results, output_dir=TABLES_DIR, window_label=window)

    print(f"\n── {window}: Temporal ──")
    display(results["temporal"])
    print(f"── {window}: Volume ──")
    display(results["volume"])
    print(f"── {window}: Frequency ──")
    display(results["frequency"])
    if results["ratio"] is not None:
        print(f"── {window}: Ratio ──")
        display(results["ratio"])

In [ ]:
# Cross-window correlation summary
cross_corr = build_cross_window_correlation_summary(all_corr_results)

for key, df_cw in cross_corr.items():
    if df_cw.empty:
        continue
    df_cw.to_csv(TABLES_DIR / f"corr_{key}_all_windows.csv", index=False)

print("Cross-window temporal summary:")
display(cross_corr["temporal"])

---
## Step 6 — Quick visualisations (for sanity-check)

Lightweight plots to verify results look sensible before handing off to Person 3.

In [ ]:
# ── Jaccard heatmap per window ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm

available_windows = [w for w in WINDOW_LABELS if w in set_reports]
if not available_windows:
    print("No set-intersection data available yet — run Step 2 first.")
else:
    fig, axes = plt.subplots(
        1, len(available_windows),
        figsize=(5 * len(available_windows), 4),
        squeeze=False,
    )

    for ax_idx, window in enumerate(available_windows):
        report = set_reports[window]
        df_active = report["active_pairwise"]

        # Build a 3×3 Jaccard matrix
        j_mat = pd.DataFrame(0.0, index=CHAINS, columns=CHAINS)
        np.fill_diagonal(j_mat.values, 1.0)
        for _, row in df_active.iterrows():
            j_mat.loc[row["chain_a"], row["chain_b"]] = row["jaccard"]
            j_mat.loc[row["chain_b"], row["chain_a"]] = row["jaccard"]

        ax = axes[0][ax_idx]
        im = ax.imshow(j_mat.values, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(CHAINS)))
        ax.set_yticks(range(len(CHAINS)))
        ax.set_xticklabels(CHAINS, rotation=45, ha="right")
        ax.set_yticklabels(CHAINS)
        ax.set_title(f"Active Jaccard\n{window}")
        for i in range(len(CHAINS)):
            for j in range(len(CHAINS)):
                ax.text(j, i, f"{j_mat.values[i, j]:.4f}", ha="center", va="center",
                        fontsize=8, color="black")

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "p2_jaccard_heatmaps.png", bbox_inches="tight")
    plt.show()
    print("Saved:", FIGURES_DIR / "p2_jaccard_heatmaps.png")

In [ ]:
# ── Temporal delta histograms for the smallest available window ───────────────
from correlation import temporal_delta_analysis

feat_window = available_windows[0] if available_windows else None
if feat_window:
    feat_csv = FEATURE_ROW_DIR / f"address_chain_features_{feat_window}.csv"
    if feat_csv.exists():
        df_f = pd.read_csv(feat_csv)
        pairs = [(CHAINS[i], CHAINS[j]) for i in range(len(CHAINS)) for j in range(i+1, len(CHAINS))]

        fig, axes = plt.subplots(1, len(pairs), figsize=(6*len(pairs), 4), squeeze=False)
        for idx, (ca, cb) in enumerate(pairs):
            result = temporal_delta_analysis(df_f, ca, cb)
            deltas = result.get("deltas_days", np.array([]))
            ax = axes[0][idx]
            if len(deltas) > 0:
                ax.hist(deltas, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
            ax.set_title(f"Δ first-seen (days)\n{ca} vs {cb} [{feat_window}]")
            ax.set_xlabel("Days (positive = chain_a first)")
            ax.set_ylabel("Addresses")
            ax.axvline(0, color="red", linestyle="--", alpha=0.7)

        plt.tight_layout()
        fig.savefig(FIGURES_DIR / f"p2_temporal_deltas_{feat_window}.png", bbox_inches="tight")
        plt.show()
    else:
        print(f"Feature CSV not found for {feat_window}")

In [ ]:
# ── Volume scatter plots ──────────────────────────────────────────────────────
from correlation import volume_correlation

if feat_window and (FEATURE_ROW_DIR / f"address_chain_features_{feat_window}.csv").exists():
    df_f = pd.read_csv(FEATURE_ROW_DIR / f"address_chain_features_{feat_window}.csv")
    pairs = [(CHAINS[i], CHAINS[j]) for i in range(len(CHAINS)) for j in range(i+1, len(CHAINS))]

    fig, axes = plt.subplots(1, len(pairs), figsize=(6*len(pairs), 4), squeeze=False)
    for idx, (ca, cb) in enumerate(pairs):
        res = volume_correlation(df_f, ca, cb)
        v_a = res.get("v_a", np.array([]))
        v_b = res.get("v_b", np.array([]))
        ax = axes[0][idx]
        if len(v_a) > 0:
            ax.scatter(np.log1p(v_a), np.log1p(v_b), alpha=0.5, s=15, color="steelblue")
        ax.set_xlabel(f"log(1 + tx_count) [{ca}]")
        ax.set_ylabel(f"log(1 + tx_count) [{cb}]")
        r = res.get("pearson_r_log")
        ax.set_title(
            f"Volume correlation\n{ca} vs {cb}  r={r:.3f}" if r is not None
            else f"Volume correlation\n{ca} vs {cb}"
        )

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f"p2_volume_scatter_{feat_window}.png", bbox_inches="tight")
    plt.show()

---
## Step 7 — Export summary for Person 3

Verify all expected output files exist.

In [ ]:
# ── Step 7: Check that all output files were created ─────────────────────────
expected_files = []

for window in WINDOW_LABELS:
    expected_files += [
        TABLES_DIR / f"set_intersections_active_{window}.csv",
        TABLES_DIR / f"set_intersections_passive_{window}.csv",
        TABLES_DIR / f"set_intersections_mixed_{window}.csv",
        TABLES_DIR / f"set_intersections_summary_{window}.csv",
        TABLES_DIR / f"classified_{window}.csv",
        TABLES_DIR / f"corr_temporal_{window}.csv",
        TABLES_DIR / f"corr_volume_{window}.csv",
        TABLES_DIR / f"corr_frequency_{window}.csv",
    ]

expected_files += [
    TABLES_DIR / "set_intersections_all_windows.csv",
    TABLES_DIR / "classification_summary_all_windows.csv",
    TABLES_DIR / "p2_feature_validation.csv",
]

print(f"{'STATUS':<8} {'FILE'}")
print("-" * 70)
all_ok = True
for f in expected_files:
    status = "OK" if f.exists() else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"{status:<8} {f.relative_to(REPO_ROOT)}")

print()
print("All outputs present:" if all_ok else "WARNING: some outputs are missing — check above")